# One current sheet per run: $J^2$ across it

For each run, $J^2$ from the Gaussian-filtered B, the cut along its steepest descent through the chosen sheet, and
the three cuts together. (The former notebook repeated the same cells for each run.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pta import config
from pta.fields import current_density
from pta.filtering import gaussian_fft_filter_2d
from pta.plotting import save_figure, show_field, use_style
from pta.sheets import descent_direction, line_profile

use_style()
RESULTS = config.results_dir("single_sheet", "SingleCSBetaComparison")

In [ ]:
SNAPSHOT_TIME = 40
# per run: the Gaussian filter width (fraction of the number of cells) and the cell of the chosen J^2 maximum
CASES = {
    16: dict(filter_fraction=0.1, sheet=(3680, 2388)),
    1: dict(filter_fraction=0.2, sheet=(3732, 2228)),
    0.0625: dict(filter_fraction=0.2, sheet=(3731, 2294)),
}
CUT_HALF_LENGTH = 70                  # cells
CUT_ZOOM_HALF_BOX = 500

In [ ]:
cuts = {}
for beta_p, case in CASES.items():
    run = config.run("beta_scan", beta_p=beta_p)
    dx, dy = run.dx, run.dy
    filter_width = case["filter_fraction"] * run.shape[0]
    B = (gaussian_fft_filter_2d(component, k0=filter_width) for component in run.load_many(["Bx", "By", "Bz"], SNAPSHOT_TIME))
    Jx, Jy, Jz = current_density(*B, dx, dy)
    J2 = Jx**2 + Jy**2 + Jz**2
    sheet = case["sheet"]
    direction = descent_direction(J2, sheet)
    s, profile = line_profile(J2, sheet, direction, CUT_HALF_LENGTH)
    cuts[beta_p] = (s * dx, profile)
    when = rf"$\beta_p = {beta_p}$, t = {SNAPSHOT_TIME}"

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(s * dx, profile)
    ax.axvline(0, linestyle="--")
    ax.set_xlabel(r"Distance along the cut ($d_p$)")
    ax.set_ylabel(r"$J^2$")
    ax.set_title(r"$J^2$ along the steepest descent, " + when, fontsize=10)
    save_figure(fig, RESULTS, f"J2_cut_beta_p{beta_p}")
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 6))
    i0, j0 = max(sheet[0] - CUT_ZOOM_HALF_BOX, 0), max(sheet[1] - CUT_ZOOM_HALF_BOX, 0)
    show_field(ax, J2[i0:sheet[0] + CUT_ZOOM_HALF_BOX, j0:sheet[1] + CUT_ZOOM_HALF_BOX], dx, dy, label=r"$J^2$",
               corner=(i0 * dx, j0 * dy))
    x, y = (sheet[0] + 0.5) * dx, (sheet[1] + 0.5) * dy
    ax.plot(x + s * direction[0] * dx, y + s * direction[1] * dy, color="yellow", linewidth=2)
    ax.scatter([x], [y], color="white", marker="x")
    ax.set_title("The cut, " + when, fontsize=10)
    save_figure(fig, RESULTS, f"J2_cut_zoom_beta_p{beta_p}")
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for beta_p, (distance, profile) in cuts.items():
    ax.plot(distance, profile, label=rf"$\beta_p={beta_p}$")
ax.axvline(0, linestyle="--", color="k")
ax.set_xlabel(r"Distance along the cut ($d_p$)")
ax.set_ylabel(r"$J^2$")
ax.set_title(r"$J^2$ along the steepest descent")
ax.legend()
save_figure(fig, RESULTS, "J2_cuts_all_beta_p")
plt.show()